In [5]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import models, layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import classification_report

In [6]:
df=pd.read_csv("cleaned_dataset_taiwan_2months.csv")
df.head()

,date,sitename,county,aqi,status,so2,co,o3,pm10,pm2.5,no2,nox,no,siteid
0,2024-08-31 23:00:00,Hukou,Hsinchu County,62.0,Moderate,0.9,0.17,35.0,18.0,17.0,2.3,2.6,0.3,22
1,2024-08-31 23:00:00,Zhongming,Taichung City,50.0,Good,1.6,0.32,27.9,27.0,14.0,7.6,9.3,1.6,31
2,2024-08-31 23:00:00,Zhudong,Hsinchu County,45.0,Good,0.4,0.17,25.1,21.0,13.0,2.9,4.1,1.1,23
3,2024-08-31 23:00:00,Hsinchu,Hsinchu City,42.0,Good,0.8,0.20,30.0,19.0,10.0,4.0,4.8,0.7,24
4,2024-08-31 23:00:00,Toufen,Miaoli County,50.0,Good,1.0,0.16,33.5,18.0,14.0,1.8,3.1,1.2,25


In [7]:
features = ['aqi', 'so2', 'co', 'o3', 'pm10', 'pm2.5', 'no2', 'nox', 'no']
X = df[features]
y = df['status']

In [8]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
# This explicitly defines the input so feature extraction works correctly
inputs = layers.Input(shape=(X_train_scaled.shape[1],))
x = layers.Dense(64, activation='relu')(inputs)
x = layers.Dropout(0.2)(x)
# This layer will be our feature source for the SVM
feature_layer = layers.Dense(32, activation='relu')(x)
outputs = layers.Dense(len(le.classes_), activation='softmax')(feature_layer)

In [11]:
model = models.Model(inputs=inputs, outputs=outputs)

In [12]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train_scaled, y_train, epochs=10, batch_size=32,validation_split=0.1)

Epoch 1/10
2834/2834 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9789 - loss: 0.0701 - val_accuracy: 0.9917 - val_loss: 0.0221
Epoch 2/10
2834/2834 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9923 - loss: 0.0198 - val_accuracy: 0.9969 - val_loss: 0.0099
Epoch 3/10
2834/2834 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.9950 - loss: 0.0135 - val_accuracy: 0.9977 - val_loss: 0.0075
Epoch 4/10
2834/2834 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.9960 - loss: 0.0103 - val_accuracy: 0.9981 - val_loss: 0.0054
Epoch 5/10
2834/2834 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.9965 - loss: 0.0095 - val_accuracy: 0.9973 - val_loss: 0.0061
Epoch 6/10
2834/2834 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.9971 - loss: 0.0079 - val_accuracy: 0.9968 - val_loss: 0.0085
Epoch 7/10
2834/2834 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.9975 - loss: 0.0072 - val_accuracy: 0.9971 - val_loss: 0.0064
Epoch 8/10
2834/2834 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.9971 - loss: 0.0075 - 

In [13]:
# Extract features from the trained neural network (using the second-to-last layer)
feature_extractor = models.Model(inputs=model.input, outputs=model.layers[-2].output)
X_train_features = feature_extractor.predict(X_train_scaled)
X_test_features = feature_extractor.predict(X_test_scaled)

# Train SVM on the extracted features
svm_model = SVC(kernel='rbf', probability=True)
svm_model.fit(X_train_features, y_train)

# Evaluate SVM
y_pred_svm = svm_model.predict(X_test_features)
print(classification_report(y_test, y_pred_svm, target_names=le.classes_))

3148/3148 ━━━━━━━━━━━━━━━━━━━━ 3s 883us/step
787/787 ━━━━━━━━━━━━━━━━━━━━ 1s 905us/step
                                precision    recall  f1-score   support

                          Good       1.00      1.00      1.00     22982
                      Moderate       1.00      1.00      1.00      2173
Unhealthy for Sensitive Groups       1.00      1.00      1.00        28

                      accuracy                           1.00     25183
                     macro avg       1.00      1.00      1.00     25183
                  weighted avg       1.00      1.00      1.00     25183



In [14]:
from sklearn.metrics import accuracy_score, precision_score, f1_score
from tensorflow.keras.callbacks import EarlyStopping

# Compute global metrics for the SVM model
global_accuracy = accuracy_score(y_test, y_pred_svm)
global_precision = precision_score(y_test, y_pred_svm, average='weighted')
global_f1 = f1_score(y_test, y_pred_svm, average='weighted')

print(f"Global Accuracy: {global_accuracy}")
print(f"Global Precision: {global_precision}")
print(f"Global F1-Score: {global_f1}")

# Add a callback for future model training (e.g., EarlyStopping)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

Global Accuracy: 0.9999205813445579
Global Precision: 0.9999205813445579
Global F1-Score: 0.9999205813445579


In [15]:
import joblib

# Save the SVM model to a file
#joblib.dump(svm_model, 'svm_model.pkl')

In [16]:
joblib.dump(scaler,'scaler.pkl')
joblib.dump(feature_extractor,'feature_extractor.h5')

['feature_extractor.h5']

In [19]:
print(list(enumerate(le.classes_)))

[(0, 'Good'), (1, 'Moderate'), (2, 'Unhealthy for Sensitive Groups')]


[1 0 0 ... 0 0 0]
